In [1]:
"""Train the SOD detector and scale its anomaly scores."""

import os
import sys

# Clear any cached modules
if 'config' in sys.modules:
    del sys.modules['config']
if 'to_tflite' in sys.modules:
    del sys.modules['to_tflite']

import pandas as pd
from pyod.models.sod import SOD
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import config
import to_tflite

# Build the detector from its configuration.
sod = SOD(
    contamination=config.CONTAMINATION,
    n_neighbors=config.N_NEIGHBORS,
    ref_set=config.REF_SET,
    alpha=config.ALPHA,
)

# NOTE: all dataset features must be pre-normalized so that, after the int8
# quantization in keras/to_tflite.py, no value overflows the int8 range.
# Fit the configured features from the CSV.
features = pd.read_csv(config.CSV_PATH)[config.FEATURES].to_numpy()
X_train, X_test = train_test_split(
    features, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE
)
sod.fit(X_train)

# Scale the raw anomaly scores into the configured range.
raw_scores = sod.decision_function(X_train).reshape(-1, 1)
score_scaler = MinMaxScaler(feature_range=config.SCORE_RANGE).fit(raw_scores)
train_scores = score_scaler.transform(raw_scores).ravel()
test_scores = score_scaler.transform(
    sod.decision_function(X_test).reshape(-1, 1)
).ravel()

# Learn the scores with a small network and export .keras + .tflite.
to_tflite.to_tflite(
    name="sod",
    x_train=X_train,
    y_train=train_scores,
    x_test=X_test,
    y_test=test_scores,
    layers=config.LAYERS,
    learning_rate=config.LEARNING_RATE,
    epochs=config.EPOCHS,
    batch_size=config.BATCH_SIZE,
    verbose=config.VERBOSE,
    feature_names=config.FEATURES,
)


2026-08-14 18:14:24.241559: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Epoch 1/50


2026-08-14 18:52:20.693197: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


3747/3747 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 3.6702e-04 - val_loss: 0.0015
Epoch 2/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - loss: 3.6667e-04 - val_loss: 0.0015
Epoch 3/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 3.6591e-04 - val_loss: 0.0015
Epoch 4/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 3.6608e-04 - val_loss: 0.0016
Epoch 5/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 3.6610e-04 - val_loss: 0.0015
Epoch 6/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 3.6561e-04 - val_loss: 0.0016
Epoch 7/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 3.6587e-04 - val_loss: 0.0015
Epoch 8/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 3.6520e-04 - val_loss: 0.0016
Epoch 9/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - loss: 3.6556e-04 - val_loss: 0.0015
Epoch 10/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 10s 1ms/step - loss: 3.6524e-04 - val_loss: 0.0015
Epoch 11/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 3.6553e-04 - val_loss: 0.0

INFO:tensorflow:Assets written to: /tmp/tmpaleo84g2/assets


Saved artifact at '/tmp/tmpaleo84g2'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 2), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  139816622684880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139816622679888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139816149783120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139816149783504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139816149782352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139816149781008: TensorSpec(shape=(), dtype=tf.resource, name=None)


/home/patrik/Diplomka/LODA/env/lib/python3.12/site-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1786726605.588103   18531 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1786726605.588118   18531 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2026-08-14 18:56:45.588320: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpaleo84g2
2026-08-14 18:56:45.588584: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-08-14 18:56:45.588588: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpaleo84g2
I0000 00:00:1786726605.593639   18531 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
2026-08-14 18:56:45.593973: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-08-14 1

<Sequential name=sequential, built=True>